## Import

In [ ]:
import os
import gc
import sys
import warnings
import numpy as np
import pandas as pd
from glob import glob
import pickle

sys.path.append("../input/pythonbox")
from box import Box

# Torch
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset 
# from torchvision.models import efficientnet_v2_l

# Lightning
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, seed_everything

In [ ]:
!pip uninstall timm --yes

sys.path.append("/kaggle/input/timmmaster")
# import timm
from timm import create_model

# print(timm.__version__)

## Config

In [ ]:
config = {'exp_name':'exp_013',
          'root': '../input/petfinder-pawpularity-score/',  # Data root
          'seed': 2023,
          'n_splits': 5,
          'image_size': 400,
          'model':{
              'package': 'timm',  # timm or torchvision
              'name': 'convnextv2_base',
              'output_dim': 1,
              'pretrain': False,
          },
          'save_dir': 'convnextv2_base',  # 儲存權重與log的資料夾
          'test_loader': {
              'batch_size': 32,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': False,
              'drop_last': False
          },
}

config = Box(config)

## Fix Seed

In [ ]:
seed_everything(config.seed)

## Tools

In [ ]:
def RMSE(predict,target):
    return torch.sqrt(nn.MSELoss()(predict.float(), target.float()))

## Dataset

In [ ]:
class PetfinderDataset(Dataset):
    """Dataset
    Args:
        df: the dataframe from csv, and the "Id" column needs to be the path of Image
    """
    def __init__(self, df, transform=None, image_size=224):
        
        self._X = df["Id"].values
        self._y = None
        self.transform = transform
        
        # 判斷有沒有分數
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
            
    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self.transform(image)
        
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

## Model

In [ ]:
class Model(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)  # 儲存超參數
        
        self.__build_model()
        
    def __build_model(self):
        if self.hparams.model.package == 'timm':
            self.backbone = create_model(self.hparams.model.name,
                                         pretrained=False, 
                                         num_classes=0, 
                                         in_chans=3)
            num_features = self.backbone.num_features
        
        elif self.hparams.model.package == 'torchvision':
            # weights = 'DEFAULT' if self.hparams.model.pretrain else None
            self.backbone = eval(self.hparams.model.name)(weights=None)
            num_features = 1000
            
        self.fc = nn.Sequential(nn.Dropout(0.5), 
                                nn.Linear(num_features, 
                                          self.hparams.model.output_dim))

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def predict_step(self, batch, batch_idx):
        images, targets = batch
        pred = self(images).sigmoid()  # return後會自動轉成numpy
        if pred.dim == 3:
            pred = pred.squeeze()
        return pred, targets


## Test

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB

test_transform = T.Compose([T.Resize(config.image_size),
                            T.CenterCrop([config.image_size, config.image_size]),
                            T.ConvertImageDtype(torch.float),
                            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

stage = 'train'
   
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df["Id"] = df["Id"].apply(lambda x: os.path.join(config.root, stage, x + ".jpg")) # 將ID改成圖片路徑
df["Pawpularity"] = df["Pawpularity"].astype(float).apply(lambda x: x / 100.) # 將Pawpularity軟換到[0, 1]

### kFold predictions

In [ ]:
warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

loss_list = []
# fold = 0
# if True:
for fold in range(config.n_splits):
    print(f"\n{f' Fold {fold} ':=^50}")
    
    # 讀取list of indexs，並取出本次fold資料分配
    with open('dataset_temp.pickle', 'rb') as f:
        all_splits = pickle.load(f)

    train_indexes, val_indexes = all_splits[fold]
    valid_df = df.iloc[val_indexes]

    # df to dataset
    val_data = PetfinderDataset(valid_df, test_transform, config.image_size)
    predict_loader = DataLoader(val_data, **config.test_loader)
    
    model_weight = glob(f'../input/{config.save_dir}/fold_{fold}/version_0/*.ckpt')[0]

    model = Model(config).load_from_checkpoint(model_weight, map_location='cuda:0')

    trainer = pl.Trainer(logger=False, precision='16-mixed')
    results = trainer.predict(model, dataloaders=predict_loader)
    
    predictions, targets = [], []
    for prediction, target in results:
        predictions.append(prediction.numpy())
        targets.append(target.numpy())

    predictions = np.concatenate(predictions).flatten()
    targets = np.concatenate(targets).flatten()

    loss = RMSE(torch.tensor(predictions), torch.tensor(targets)).item() * 100.
    
    print(f"RMSE: {loss}")
    
    loss_list.append(loss)
    
    # 清除變數與快取
    del model
    torch.cuda.empty_cache()
    gc.collect()

### 計算平均預測分數

In [ ]:
mean_loss = np.array(loss_list).mean(axis=0)
print(mean_loss)